# Phase 2 — Extract activations and train linear probes

Mean-pools the residual stream over the *response* tokens for a set of candidate
layers, caches those features, then fits one logistic-regression probe per layer
and reports AUROC on the held-out eval set.

**Inputs:** `data/probe_train.json`, `data/probe_eval.json` (from
[`build_data.ipynb`](build_data.ipynb)).
**Outputs:** `data/features_train.npz`, `data/features_eval.npz`,
`data/baseline_training_summary.json`.

The feature-extraction step is the expensive one (a forward pass per example) and
is left commented out — it already produced the cached `.npz` files, and the
training section below loads straight from that cache.

## Setup

Shared helpers live in [`common.py`](common.py). Select the `interp` conda
environment as this notebook's kernel — it has `transformer_lens`, `torch`,
`scikit-learn` and `matplotlib` installed.

In [ ]:
import os
import sys
from pathlib import Path

# These notebooks use repo-relative paths ("data/...", "plots/..."), exactly as the
# original scripts did when run from the project root. So make that the working
# directory, and put src/ on the import path so `common` is importable.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()), Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import common
common.ensure_dirs()
print("Working directory:", Path.cwd())

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

torch.set_num_threads(os.cpu_count())
print("torch threads:", torch.get_num_threads())

## Feature extraction

Three variants: one layer at a time (for quick pilots), all candidate layers in a
single forward pass, and a batched version.

`common.get_activation` does the single-layer, single-example case.

In [ ]:
def build_features_single(model, dataset, layer):
    # Single layer — use for pre-test
    X, y = [], []
    for i, entry in enumerate(dataset):
        vec = common.get_activation(model, entry["prompt"], entry["response"], layer)
        X.append(vec.float().numpy())
        y.append(1 if entry["label"] == "deceptive" else 0)
        print(f"Processed entry {i+1}/{len(dataset)}")
    return np.array(X), np.array(y)

In [ ]:
def build_features_multilayer(model, dataset, layers):
    features = {layer: [] for layer in layers}
    y = []
    for i, entry in enumerate(dataset):
        full_text = entry["prompt"] + entry["response"]
        tokens = model.to_tokens(full_text)
        prompt_tokens = model.to_tokens(entry["prompt"])
        response_start = prompt_tokens.shape[1]

        with torch.no_grad():
            _, cache = model.run_with_cache(
                tokens,
                names_filter=lambda name: any(f"blocks.{l}.hook_resid_post" == name for l in layers)
            )

        for layer in layers:
            resid = cache[f"blocks.{layer}.hook_resid_post"][0]
            response_resid = resid[response_start:]
            features[layer].append(response_resid.mean(dim=0).float().numpy())

        y.append(1 if entry["label"] == "deceptive" else 0)
        print(f"Processed entry {i+1}/{len(dataset)}")

    y = np.array(y)
    return {layer: np.array(features[layer]) for layer in layers}, y

In [ ]:
def build_features_batched(model, dataset, layers, batch_size=8):
    features = {layer: [] for layer in layers}
    y = []
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(dataset) - 1)//batch_size + 1}")
        full_texts = [e["prompt"] + e["response"] for e in batch]
        tokens = model.to_tokens(full_texts, padding_side="right")  # pads to longest in batch

        with torch.no_grad():
            _, cache = model.run_with_cache(
                tokens,
                names_filter=lambda name: any(f"blocks.{l}.hook_resid_post" == name for l in layers)
            )

        for j, entry in enumerate(batch):
            prompt_len = model.to_tokens(entry["prompt"]).shape[1]
            for layer in layers:
                resid = cache[f"blocks.{layer}.hook_resid_post"][j]
                features[layer].append(resid[prompt_len:].mean(dim=0).float().numpy())
            y.append(1 if entry["label"] == "deceptive" else 0)

        print(f"[{i+len(batch)}/{len(dataset)}] done")

    return {l: np.array(v) for l, v in features.items()}, np.array(y)

## Pilot: how much data does the probe need?

A learning curve on a single mid-early layer, used to sanity-check that the
dataset size is in a sensible range before committing to the full extraction.

In [ ]:
def train_and_eval_probe(X, y, test_size=0.3):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    clf = common.fit_probe(X_train, y_train, C=1.0)
    y_scores = clf.predict_proba(X_test)[:, 1]
    auroc = roc_auc_score(y_test, y_scores)
    return clf, auroc


def learning_curve(X, y, sizes, n_repeats=5):
    results = {}
    for n in sizes:
        aurocs = []
        for seed in range(n_repeats):
            idx = np.random.RandomState(seed).choice(len(y), size=min(n, len(y)), replace=False)
            X_sub, y_sub = X[idx], y[idx]
            try:
                _, auroc = train_and_eval_probe(X_sub, y_sub, test_size=0.3)
                aurocs.append(auroc)
            except ValueError:
                continue  # skip if a class is missing in a split
        results[n] = aurocs
        print(f"n={n}: mean AUROC={np.mean(aurocs):.3f}, std={np.std(aurocs):.3f}")
    return results


def plot_learning_curve(results):
    sizes = sorted(results.keys())
    means = [np.mean(results[n]) for n in sizes]
    stds = [np.std(results[n]) for n in sizes]

    plt.figure(figsize=(7, 5))
    plt.errorbar(sizes, means, yerr=stds, marker='o', capsize=4)
    plt.axhline(0.5, color='gray', linestyle='--', label='Chance (AUROC=0.5)')
    plt.xlabel("Number of training examples")
    plt.ylabel("AUROC")
    plt.title("Probe AUROC vs. dataset size")
    plt.ylim(0.4, 1.05)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("data/learning_curve.png", dpi=150)

In [ ]:
def small_test_paper(model):
    # Load a small dataset
    with open("data/instructed_generated.json") as f:
        pilot_instructed_results_as_list = json.load(f)

    layer = model.cfg.n_layers // 4
    X, y = build_features_single(model, pilot_instructed_results_as_list, layer=layer)
    if len(X) == 0 or len(y) == 0:
        print("Wrong data, please check")
        return
    print("Building complete.")

    # Save features
    np.savez(f"data/features_layer{layer}.npz", X=X, y=y)
    print(f"Saved {X.shape[0]} examples, dim={X.shape[1]}")

    # Estimate converging size
    loaded = np.load(f"data/features_layer{layer}.npz")
    X_load, y_load = loaded["X"], loaded["y"]
    sizes = [20, 40, 60, 80, 100, 120, 150]
    results = learning_curve(X_load, y_load, sizes)
    plot_learning_curve(results)
    return results

## Load the model

Only needed for feature extraction — skip it if you are loading cached features
below. Pinned to CPU/bfloat16, matching how the cached features were extracted.

In [ ]:
model = common.load_model(device="cpu", dtype=torch.bfloat16)

### Run the pilot

⚠️ Slow (one forward pass per example). Uncomment to run.

In [ ]:
# small_test_paper(model)

## Full feature extraction

⚠️ Slow, and **overwrites** `data/features_train.npz` / `data/features_eval.npz`.
This is the step that produced the cached features — uncomment only to
regenerate them.

In [ ]:
def generate_features_production(model):
    n_layers = model.cfg.n_layers
    candidate_layers = list(range(2, n_layers, 3))

    with open("data/probe_train.json") as f:
        train_data = json.load(f)
    with open("data/probe_eval.json") as f:
        eval_data = json.load(f)

    print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
    print("Candidate layers:", candidate_layers)

    X_train_by_layer, y_train = build_features_multilayer(model, train_data, candidate_layers)
    X_eval_by_layer, y_eval = build_features_multilayer(model, eval_data, candidate_layers)

    np.savez("data/features_train.npz", y=y_train,
             **{f"X_layer{l}": X_train_by_layer[l] for l in candidate_layers})
    np.savez("data/features_eval.npz", y=y_eval,
             **{f"X_layer{l}": X_eval_by_layer[l] for l in candidate_layers})

    return candidate_layers, n_layers

In [ ]:
# generate_features_production(model)

## Train and evaluate one probe per layer

Loads the cached features, so this runs without the extraction step above.

In [ ]:
def train_and_evaluate_probes(X_train_by_layer, y_train, X_eval_by_layer, y_eval,
                              n_layers=None, C=1.0):
    results_per_layer = {}

    for layer in sorted(X_train_by_layer):
        clf = common.fit_probe(X_train_by_layer[layer], y_train, C=C)
        y_scores = clf.predict_proba(X_eval_by_layer[layer])[:, 1]
        auroc = roc_auc_score(y_eval, y_scores)
        results_per_layer[layer] = auroc
        print(f"Layer {layer}: AUROC = {auroc:.3f}")

    summary = {
        "n_layers_total": n_layers,
        "train_size": len(y_train),
        "eval_size": len(y_eval),
        "train_deceptive": int(sum(y_train)),
        "eval_deceptive": int(sum(y_eval)),
        "results_per_layer": {str(layer): auroc for layer, auroc in results_per_layer.items()},
    }

    with open("data/baseline_training_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary

In [ ]:
X_train_by_layer, y_train, X_eval_by_layer, y_eval = common.load_features()

In [ ]:
# n_layers_total is summary metadata; None if the model isn't loaded in this session.
n_layers = model.cfg.n_layers if "model" in globals() else None

summary = train_and_evaluate_probes(
    X_train_by_layer, y_train, X_eval_by_layer, y_eval, n_layers=n_layers
)

best_layer = max(summary["results_per_layer"], key=summary["results_per_layer"].get)
print(f"\nBest layer: {best_layer} (AUROC {summary['results_per_layer'][best_layer]:.3f})")